# CoordAnalyst - an educational tool to analyse coordination complexes


Claire Bachmann, Bertille Delloye, Enéa Drezet--Marçot, Victor Davril


## Introduction

Coordination chemistry is an active field of research, with its study sitting at the intersection of many different branches of science, and its impact being major. In medicine, coordination compounds are indispensable. Cisplatin, one of the most widely used anticancer drugs in the world, is a simple platinum complex discovered in the 1960s that remains a frontline treatment for testicular, ovarian, and lung cancers. Another exmaple is Hemoglobin, the protein that carries oxygen through our blood, an iron coordination complex. In catalysis and industry, coordination chemistry drives some of the most economically significant chemical processes on the planet.The production of ammonia, acetaldehyde or even plastics, using the Haber-Bosch process, the Wacker process, and Ziegler-Natta catalysts, relies on iron, palladium and aluminium catalysts. In materials science, coordination compounds called metal-organic frameworks (MOFs) have emerged as one of the most exciting research areas of the 21st century, with applications in gas storage, carbon capture, drug delivery, and chemical sensing. Their extraordinary surface areas and tunable pore geometries make them uniquely powerful materials. In biology, nearly a third of all known proteins contain a metal ion at their active site : these coordination complexes perform catalysis, electron transfer, or structural stabilisation. Understanding these metalloenzymes is central to drug design and our understanding of life itself.  
Despite this enormous importance, coordination compounds remain challenging to characterise. Infrared and Raman spectroscopy are among the primary experimental tools used to study them, in order to reveal coordination modes, identify ligands, confirm geometries, and monitor reactions in real time. Yet interpreting these spectra requires deep chemical knowledge and access to reference data that is scattered across literature.  
Coordination chemistry is taught as a Bachelor's course in chemistry as the concepts are extremely relevant, and sometimes intertwine with spectroscopy and organometallic chemistry. Students are expected to aquire an understanding of metal and ligands, oxidation states, geometry, d-electrons, and bond stretches.    
Our package provides an educational tool with which students can try out different complexes' names and formulas and verify their properties, as well as visualize them in two and three dimensions. Moreover, CoordAnalyst provides an approximate IR and Raman spectra of the complex based on reference data from Nakamoto tables. It is presented as a unified interface, designed to make the spectral characterisation of coordination compounds more accessible to students and academics.



## Material and methods

### Software Architecture

The project is organised as a Python package, with the main source code stored in `src/coordchem`. The data used by the program is kept outside the main source code, so that the package functions and the chemical data remain separated. Inside `coordchem`, the core files contain the general functions used to interpret a coordination complex. The file `parser.py` defines `parse_formula()`, which extracts the metal, ligands, charge, counter-ions, oxidation state and coordination number from a formula. The file `name.py` defines `parse_name()`, which performs a similar task starting from a coordination compound name. Both parsers return the same structured object, `ParsedComplex`. The `complex.py` file then uses these parsers to build a higher-level `Complex` object through methods such as `Complex.from_formula()`, `Complex.from_name()` and `Complex.from_input()`. This class gives direct access to useful properties and links the parsed complex to geometry, visualisation and spectra functions.

The `geometry.py` file adds the chemical interpretation step. It contains functions such as `get_d_count()`, `get_geometry()`, `predict_geometry()` and `geometry_report()`. These functions use the parsed information to calculate the metal d-electron count and predict an idealised coordination geometry from the coordination number, metal, oxidation state and ligand information.

Two subpackages were created inside `coordchem`: `viz` and `spectra`. The `viz` subpackage contains the tools used to generate 2D and 3D visualisations. For 2D drawing, `ligand_data.py` stores ligand SMILES strings, donor atom indices and display labels. The file `diagram_2d.py` contains the main drawing functions, including `parse_complex_input()`, `build_coordination_mol()`, `diagram_2d_svg()` and `save_diagram_2d()`. The file `layout_2d.py` defines the idealised coordination-site positions with functions such as `coordination_sites()`, `chelate_octahedral_sites()`, `edta_octahedral_sites()` and `mixed_polydentate_site_groups()`. The file `transform_2d.py` contains the mathematical transformations used to place ligands correctly, especially polydentate ligands, with functions such as `transform_monodentate()`, `transform_polydentate()`, `transform_acac()`, `transform_oxalate()` and `transform_edta()`.
The 3D visualisation ???

The `spectra` subpackage contains the tools used for IR and Raman prediction. The `predictor.py` file extracts relevant spectral bands from the database for an input `ParsedComplex` and returns them as a `PredictionResult` object, ordered by increasing wavenumber. The `renderer.py` file then converts these bands into a simulated spectrum by representing each band as a Gaussian peak and summing the peaks to produce the final plotted spectrum.

### 1.1 Formula and Name Parser, Properties Extraction

The first step of the program is to transform what the user writes into something that Python can actually understand and reuse. A coordination complex can be written in different ways: for example as a formula, such as `[Fe(CN)6]4-`, or as a name, such as `tetraamminecopper(II)`. Even though these two inputs look very different, our goal was to convert them into the same internal format. This makes the rest of the package much easier to use, because all later modules receive the same type of structured object containing the metal, ligands, charge, oxidation state, coordination number and other useful properties.

For formulas, this work is done by the `parser` module, mainly through the function `parse_formula(formula: str) -> ParsedComplex`. This function reads the formula step by step. It separates possible counter-ions from the coordination sphere, identifies the metal centre, detects the ligands inside the brackets, counts how many times each ligand appears, and reads the global charge of the complex. Once these pieces of information are found, the parser also computes derived chemical properties such as the oxidation state of the metal and the coordination number.

For names, the same idea is implemented in the `name` module with the function `parse_name(name: str) -> ParsedComplex`. Instead of reading brackets and charges directly, this function searches the text for known ligand names, metal names and oxidation states written in Roman numerals. For example, in `tetraamminecopper(II)`, the program recognises `ammine` as the ligand, `copper` as the metal, and `(II)` as the oxidation state. The result is then converted into the same `ParsedComplex` format as for formula inputs. This name parser is not intended to understand every possible IUPAC name, but it works for the coordination names covered by our ligand and metal tables. This makes it suitable for common examples used in the project while keeping the implementation understandable.

To make this easier to use, we added a higher-level interface in the `complex` module with the `Complex` class. This class acts as the main entry point of the package. The user does not need to know which parser should be used: `Complex.from_formula()`, `Complex.from_name()` and `Complex.from_input()` create the same kind of complex object from different types of input. This was useful because the visualisation, geometry and spectra modules can then work with one consistent object instead of handling many different input formats.

Once the complex has been parsed, the `geometry` module uses this information to predict the shape of the complex and its d-electron count. The functions `get_geometry(complex_input)` and `get_d_count(complex_input)` can accept a formula, a name or an already parsed object. This makes the workflow flexible: the user can start from a simple text input, and the program automatically moves from parsing to chemical interpretation.

Overall, this part of the package acts as the bridge between human chemical notation and the rest of the code. It turns formulas and names into a clean chemical object, enriches it with useful properties, and prepares it for geometry prediction, visualisation and spectral analysis.

### 1.3 Spectral Band Database
The data was extracted systematically from  Nakamoto, K. "Infrared and Raman Spectra of Inorganic and Coordination Compounds", 6th ed., Wiley, 2009, and placed in SEED_BANDS : list of tuples with (ligand, coordination, metal, spectrum_type, wn_min, wn_max, intensity, assignment, ir_active, raman_active, source), which is defined at the top of the file as a BandRecord object. The data is given grouped by ligand for easy verification, and for both IR and Raman stretches.

Then the class IRBandDB is defined :  
The __init__ method does three things in order every time:  
-Opens a connection to SQLite  
-Creates the ir_bands table if it doesn't exist yet (_create_table)  
-Seeds the table with data if it's empty (_seed)  
   
First, _create_table runs a CREATE TABLE IF NOT EXISTS SQL statement. It defines the columns: ligand, coordination mode, metal, spectrum type, wavenumber range, intensity, assignment, whether it's IR/Raman active, and the source reference. Also creates two indexes to make lookups faster.
Then, _seed inserts all the rows from the SEED_BANDS list at the top of the file. It uses executemany which inserts all rows in one operation rather than one by one. The row count is checked first so it never double-inserts.  

The main query method is get_bands.   
(a) builds a SQL query with WHERE ligand = ? and AND spectrum_type = ?. The ? placeholders are essential to they prevent SQL injection and handle special characters safely.  
(b) if a coordination filter like "terminal" is passed, it adds AND coordination = ? to the query _row_to_record which return a list of BandRecord.  
(c) if a specific metal is passed : It splits the results into two groups: rows specific to that metal (e.g. metal = "Fe") and generic rows (metal = "any"). It then keeps all the specific rows, and only adds generic rows whose assignment isn't already covered by a specific row. This is how, for example, the Fe-specific 2093 cm⁻¹ band takes priority over the generic 2100–2200 range for the same C≡N stretch.  
get_all_ligands — returns a simple list of all ligand symbols in the database, e.g. ["CN", "CO", "Cl", "NH3", ...]. Useful for checking what's covered.  
get_bands_in_range(wmin, wmax) is used for reverse lookup : allows to identify an unknown peak in an experimental spectrum.  
add_band(...) is for inserting a custom entry. Useful when you encounter a ligand not in the seed data. Takes all the same fields as the database columns and commits immediately.  
summary(ligand) loops over IR then Raman bands and prints them as a formatted table, used for debugging and demos.  

### 1.4 Spectrum Prediction - Gaussians and plotting
**predictor.py** has a class PredictionResult which contains the attributes bands, intensities, the type of spectrum, the metal, the complex formula, ligand_coverage and warnings.  
The main function is predict_spectrum which returns a PredictionResult object from a ParsedComplex object. It loops over very ligand in the complex using te query db.get_bands for the specific ligand, spectrum and metal. The band intensity is scaled and not yet normalised according to the number of ligand of the same type. The bands extracted are sorted by order of increasing wavenumber.   
   
Chemical corrections : 
- Backbonding shifts for π-accepting ligands (CN, CO, NO) : These ligands accept electron density from the metal via π-backbonding.
More electron-rich metals (low oxidation state) → more backbonding → weaker C≡N / C≡O bond → LOWER wavenumber (negative shift)
Less electron-rich metals (high oxidation state) → less backbonding → stronger bond → HIGHER wavenumber (positive shift)
- Coordination shifts — how much a band moves when a free ligand coordinates. Negative = redshift (moves to lower wavenumber).
- Selection rules — which M-L band types are IR/Raman active per geometry.  
Based on the mutual exclusion rule: In centrosymmetric complexes (Oh, D4h), a band cannot be both IR and Raman active.
+++

**renderer.py** first defines the mathematical gaussian for variables x, center, height and sigma. In build_spectrum : first we use linspace for x to create an array of evenly spaced numbers between wn_range[0] and wn_range[1]. We have decided 3000 points as the more points we have, the smoother the curve appears. For y, originally we create an array of the same size containing only zero. Then, looping over every band, a gaussian peak is added to the right y position.  
plot_spectrum uses x, y from build_spectrum and traces the figure. A vertical dotted line is also added from the top of the peak to the position on the x-axis to be able to easily read off the wavenumber of the stretch. 

### 1.4 Two-Dimensional Visualisation

After the complex has been parsed and its geometry has been predicted, the next step is to turn this chemical information into a readable drawing. This is the role of the two-dimensional visualisation module. The aim is not to generate an exact crystallographic structure, but to produce an idealised schematic view of the coordination sphere, showing the metal centre, the ligands and their approximate spatial arrangement.

The 2D drawing workflow is mainly handled in `diagram_2d.py`. The function `parse_complex_input()` is used at the beginning of the workflow so that the drawing functions can accept different input formats: a formula, a compound name or an already parsed `ParsedComplex` object. Then, `get_geometry()` provides the predicted coordination geometry, which is used to choose how the ligands should be arranged around the metal.

The positions of the coordination sites are generated in `layout_2d.py` by the function `coordination_sites(geometry, n_sites)`. Depending on the predicted geometry, this function returns predefined positions around the metal centre. Linear complexes are drawn with two opposite sites, square planar complexes are drawn as a flat cross, tetrahedral complexes use wedge and dashed bonds to suggest depth, and octahedral complexes are drawn with a simplified projection containing two vertical bonds and four surrounding bonds. For coordination number 8, the module can also generate a square antiprismatic-type layout.

The drawing also depends on ligand-specific information stored in `ligand_data.py`. This file contains the SMILES strings used to describe each ligand. A SMILES string is a compact text representation of a molecule that tells RDKit which atoms are present and how they are connected. For example, ethylenediamine (`en`) is represented as `NCCN`, oxalate (`ox`) as `[O-]C(=O)C(=O)[O-]`, and acetylacetonate (`acac`) as `CC(=O)C=C(O)C`.

These SMILES strings are converted by RDKit into molecular objects, called `Chem.Mol` objects. In this report, these objects are referred to as RDKit ligand structures. They are not experimental or crystallographic structures; they are connectivity-based molecular representations that RDKit can use to generate 2D coordinates and draw the ligand. The file `ligand_data.py` also stores donor atom indices, which tell the program which atom or atoms of the ligand bind to the metal. This is essential for placing the ligand correctly on the coordination sites.

Small monodentate ligands, such as NH3, H2O, CN or Cl, can be drawn as compact labels to keep the diagram readable. Larger or polydentate ligands, such as ethylenediamine, oxalate, acetylacetonate or EDTA, are drawn using their RDKit-generated ligand structures. This allows the program to show more of the ligand backbone when it is chemically useful, while still keeping the overall coordination diagram clear.

The layout is based on simple coordination chemistry rules. The coordination number defines an ideal reference geometry: linear for coordination number 2, trigonal planar for coordination number 3, tetrahedral or square planar for coordination number 4, octahedral for coordination number 6, and dodecahedral or square antiprismatic for coordination number 8. However, real coordination complexes are rarely perfectly ideal. Their structures can depend on metal size, d-electron count, ligand steric effects and electronic interactions. For example, four-coordinate complexes can be either tetrahedral or square planar: tetrahedral geometries reduce ligand--ligand repulsion, whereas square planar geometries are often favoured for low-spin d⁸ metal centres such as Pt(II), Pd(II) or Ni(II) with strong-field ligands. Six-coordinate complexes are commonly octahedral, but distortions such as tetragonal elongation, tetragonal compression or trigonal distortion may occur, especially for non-equivalent ligands or Jahn--Teller-active ions such as Cu(II).

In our implementation, we chose a simplified idealised model rather than trying to reproduce these distortions quantitatively. The main objective was to avoid overlap between ligands and to keep the diagram easy to interpret. For this reason, specific layout functions were added for cases that are difficult to draw with a simple generic geometry. For example, `chelate_octahedral_sites()` is used for tris-bidentate octahedral complexes, `tridentate_octahedral_sites()` is used for bis-tridentate octahedral complexes, and `edta_octahedral_sites()` provides a compact layout for EDTA-like hexadentate coordination.

A more general function, `mixed_polydentate_site_groups()`, was also added for complexes containing both polydentate and monodentate ligands. In this case, the polydentate ligands are assigned neighbouring coordination sites first, because they need several adjacent positions to bind correctly. The remaining sites are then filled by the monodentate ligands. This avoids many overlap problems and makes mixed-ligand complexes easier to read.

After the coordination sites are chosen, the ligand structures are placed onto these sites. This is done in `build_coordination_mol()`, which creates a composite RDKit molecule containing the metal and all ligands. The function `_expand_ligands()` first expands the ligand dictionary into a complete ligand list. Then, for each ligand, `_make_ligand_mol()` takes the SMILES string from `ligand_data.py` and converts it into an RDKit `Chem.Mol` object with 2D coordinates. The function `_match_donor_indices()` then identifies which atom or atoms of the ligand bind to the metal.

The ligand is then positioned using the transformation functions from `transform_2d.py`. Monodentate ligands are placed with `transform_monodentate()`, while ligands with several donor atoms are placed with `transform_polydentate()`. Some ligands required more specific treatment to make the final drawing clearer. For this reason, `transform_acac()` was added for acetylacetonate, `transform_oxalate()` for oxalate, and `transform_edta()` for EDTA. These functions do not calculate real molecular conformations; they adjust the ligand orientation, scale and position so that the final diagram remains readable and chemically meaningful.

The final drawing is generated by `diagram_2d_svg()`. This function builds the full coordination molecule, draws it with RDKit, adds a title and returns the result as an SVG image. The SVG format was chosen because it gives clean and scalable figures that can easily be included in reports or presentations. The function `save_diagram_2d()` then saves the SVG file to disk.

Bond styles were also used to make the drawings easier to understand. Normal lines represent bonds in the drawing plane, while bold wedges and dashed bonds suggest approximate three-dimensional orientation. This is especially useful for tetrahedral and octahedral drawings, where a flat representation would otherwise hide part of the spatial arrangement. However, this remains a simplified convention. The current 2D module does not explicitly distinguish stereochemical isomers such as cis/trans, mer/fac or Λ/Δ. It provides a general idealised representation of the coordination environment rather than a full stereochemical model.

Overall, the 2D visualisation module follows a clear sequence: the input is checked with `parse_complex_input()`, the geometry is predicted with `get_geometry()`, ligand-specific drawing information is taken from `ligand_data.py`, each ligand symbol is converted into a SMILES string and then into an RDKit `Chem.Mol` object, the coordination sites are generated by `coordination_sites()` or by one of the special layout functions, the ligands are positioned with the transformation functions, and the final SVG drawing is produced by `diagram_2d_svg()` or saved with `save_diagram_2d()`. This workflow allowed us to generate simple, readable and chemically informed diagrams of coordination complexes.

### 1.5 Three-Dimentional Visualisation

## Results

### 1.6 Web Application - Streamlit interface

### 1.7 Testing and Validation

The package was validated using a pytest test suite. The tests were organised by module in order to check each part of the workflow separately before testing more integrated behaviours. This was important because the package combines several steps: parsing the input, extracting chemical properties, predicting the geometry, generating visualisations and predicting spectral bands.

The parser was tested in `test_parser.py`. These tests check that formulas such as `[Fe(CN)6]4-`, `[PtCl2(NH3)2]`, `[Co(en)3]3+` and `K4[Fe(CN)6]` are correctly interpreted. They verify the extraction of the metal, ligands, complex charge, counter-ions, ligand metadata, oxidation state and coordination number. Edge cases were also tested, such as unknown ligands, missing charge suffixes, whitespace in the input and invalid metal symbols.

The name parser was tested in `test_name.py`. These tests verify that names such as `tetraamminecopper(II)`, `hexacyanoferrate(II)` and `diamminedichloroplatinum(II)` are converted into the same type of `ParsedComplex` object as formula inputs. They also check the extraction of oxidation states from Roman numerals, the recognition of metal names and the parsing of ligand prefixes such as di-, tetra- and hexa-.

The `Complex` class was tested in `test_complex.py`. These tests check that `Complex.from_formula()`, `Complex.from_name()` and `Complex.from_input()` correctly build a high-level `Complex` object. They also verify that the class exposes the expected properties, including the metal, ligands, oxidation state, coordination number, predicted geometry and d-electron count.

The geometry module was tested in `test_geometry.py`. These tests validate the rule-based geometry prediction for common textbook examples, such as linear silver complexes, octahedral hexacyanoferrate, square-planar platinum(II) complexes and ambiguous four-coordinate nickel or copper complexes. The same test file also checks the calculation of the metal d-electron count.

The 2D visualisation module was tested in `test_diagram_2d.py`. These tests check that SVG diagrams are generated correctly, that files can be saved, and that different geometries produce the expected idealised projections. More specific tests validate the treatment of chelating ligands, EDTA, mixed polydentate/monodentate complexes, compact labels for small ligands, wedge/dash bond styles and square-antiprismatic projections.

The 3D visualisation module was tested in `test_structure_3d.py`. These tests check the generation of idealised coordination-site positions, the creation of 3D ligand structures from SMILES strings, donor atom detection and the construction of a complete RDKit molecule with a 3D conformer. They also verify that the `Complex` class can access the 3D building and HTML display functions.

Finally, the spectral prediction part was tested using `test_ir_ra_bands.py` and `test_predictor.py`. These tests verify that the spectral database contains the expected ligands and band records, that IR and Raman bands can be extracted, and that the predictor returns a `PredictionResult` object with bands sorted by increasing wavenumber. Benchmark complexes such as hexacyanoferrate, cisplatin and chromium hexacarbonyl were used to check that characteristic spectral regions are represented.

Overall, the tests show that the main workflow of the package is functional: a complex can be parsed from a formula or a name, converted into chemical properties, assigned a predicted geometry, visualised in 2D or 3D, and connected to spectral prediction.

## Results

## Discussion


### 3.1 Accuracy and limitations

restricted to one-metal-center only complexes, no bridging (?)

### 3.2 Value and comparison with existing tools

### 3.3 Future Improvements

Several improvements could be made to extend the current 2D visualisation module. One important improvement would be to handle stereochemical information more explicitly. In the current version, the 2D representation does not automatically generate specific cis/trans, mer/fac or Λ/Δ isomers when the input name or formula does not specify the exact configuration. This is a deliberate limitation, because the same formula can correspond to several stereoisomers, and choosing one randomly could be chemically misleading.

However, when the stereochemistry is clearly given in the input, the package could be improved to use this information. For example, the name parser could be extended to recognise descriptors such as cis, trans, fac, mer, Λ and Δ. The 2D visualisation module could then place the ligands in the correct relative positions and draw the requested stereoisomer explicitly. This would make the package more accurate while keeping the default behaviour safe: unspecified structures would remain idealised, while specified stereochemical names would produce the corresponding configuration. This would require not only changes in the 2D drawing module, but also an extension of the name parser so that stereochemical descriptors are stored in the parsed object and passed to the visualisation functions.

Another improvement would be to represent distortions more realistically. The current layout is based on ideal geometries, such as tetrahedral, square planar or octahedral arrangements. In future versions, simplified models could be added for common distortions, such as Jahn--Teller elongation in Cu(II) complexes, tetragonal compression or trigonal distortion. This would make the drawings closer to the structures discussed in coordination chemistry.

The representation could also be improved by including more realistic metal--ligand bond distances. At the moment, the bond lengths in the 2D diagrams are chosen mainly for readability and to avoid overlap. A future version could use approximate bond distances from experimental data or from a curated database, depending on the metal, oxidation state and donor atom. This would make the diagrams more chemically informative, while still keeping them schematic.

Finally, the export options could be improved. The module currently generates SVG drawings, which are useful because they are clean and scalable. Direct export to PNG or PDF, together with more control over figure size, labels and bond style, would make the visualisation module easier to use in reports, presentations and teaching material.

# Conclusion